# InDels outside tandem repeats

The 1-3 bp InDels called outside repeats by `workflows/indel_mutation_call.py`.

**Input:** `results/nonrepetitive_indels/{species}/{sample}/`
- `indels.tsv`: one row per read carrying a call, after the final filters
  (haploid concordance, kmerQ >= 93). There is no read-end filter here.
- `opportunities.tsv`: read positions outside the mask with kmerQ >= 93, counted
  per contig, context, tissue and mapQ.
- `filter_report.tsv`: rows removed by each final filter.
- `indels_classified.tsv`: the calls plus the read's germline/somatic
  likelihoods, for the samples that have them.

**Every sample is loaded.** The core sections below cover all of them; the
sections after that cover one group at a time, taken from config.yaml:

| group | config list | what it adds |
|---|---|---|
| tissue | `tissue_samples` | reads carry a tissue label, so rates can be split by tissue |
| germline/somatic | `methylation_samples` | reads carry cell-type likelihoods |
| sperm | `sperm_samples` | one tissue by construction, nothing per read |

A sample can appear in more than one list; the flags are kept separately and
`group` is only the first match, for tables that need one label per sample.

**The unique flag runs through everything.** The pileup marks a position
`non-unique` when more than one read there carries an InDel of any size. Those
calls are far more often assembly artefacts than mutations, so instead of
dropping them up front, every table and plot below keeps the two apart. Note
that both classes share one denominator: the opportunities are read positions
and carry no such flag, so a "non-unique rate" is the rate at which
non-unique-flagged calls arise per read position.

In [ ]:
library(readr)
library(dplyr)
library(tidyr)
library(ggplot2)
library(yaml)

results_dir           <- "../results/nonrepetitive_indels"
repeat_indels_dir     <- "../results/repetitive_indels"  # for the comparison with the repeat models
config_path           <- "../config.yaml"                # the sample groups come from here
models                <- c("M0", "M1")  # repeat models to compare with; add "M2" to include it
species_keep          <- NULL           # e.g. c("human", "chimp"); NULL keeps every species
max_hap_distance      <- 0            # bp; 0 keeps only calls whose haploid InDel is at the same position
min_read_end_distance <- 0              # bp; the repeat calls use 100, see "Distance to the read end"

figures_dir <- "../plots"   # where save_svg() writes

options(repr.plot.width = 11, repr.plot.height = 6)

# One fixed colour per model, the same in every notebook in this folder, and a
# neutral one for the sequence outside repeats.
model_colours  <- c(outside = "#898781", M0 = "#2a78d6", M1 = "#eb6834", M2 = "#1baf7a")
type_colours   <- c(deletion = "#52514e", insertion = "#b0aea8")
unique_colours <- c(unique = "#2a78d6", `non-unique` = "#eb6834")
group_colours  <- c(tissue = "#2a78d6", `germline/somatic` = "#eb6834",
                    sperm = "#1baf7a", other = "#898781")

# Save a plot for a slide or a figure. SVG keeps the text as text, so PowerPoint
# and Illustrator can restyle it, and it stays sharp at any size. 11 x 5.5 inches
# fills a 16:9 slide with room for a title above.
save_svg <- function(plot, file, width = 11, height = 5.5) {
  dir.create(figures_dir, showWarnings = FALSE, recursive = TRUE)
  path <- file.path(figures_dir, file)
  device <- if (requireNamespace("svglite", quietly = TRUE)) svglite::svglite else grDevices::svg
  ggsave(path, plot, width = width, height = height, device = device)
  cat("wrote", path, "\n")
  invisible(path)
}

# Height for a figure with one panel per sample.
facet_height <- function(n, per_row = 4, row_height = 2.4) 3 + row_height * ceiling(n / per_row)

# The calls, with the columns that can be empty for a whole file read as text.
read_calls <- function(path) {
  read_tsv(path, show_col_types = FALSE, progress = FALSE,
           col_types = cols(tissue = col_character(), other_ops = col_character(),
                            contig_hap = col_character()))
}

# TRUE when the file is there and has at least one row under the header. Some
# samples have no calls at all, and read_tsv() then guesses every column as text,
# which does not stack with the other samples.
has_rows <- function(paths) {
  vapply(paths, function(p) {
    file.exists(p) && length(readLines(p, n = 2, warn = FALSE)) > 1
  }, logical(1), USE.NAMES = FALSE)
}

# Rate with an exact 95% Poisson interval, from columns indels and opportunities.
add_rate <- function(d, level = 0.95) {
  a <- (1 - level) / 2
  mutate(d,
         rate = indels / opportunities,
         low  = qchisq(a, 2 * indels) / 2 / opportunities,
         high = qchisq(1 - a, 2 * (indels + 1)) / 2 / opportunities)
}

# A rate of 0 cannot go on a log axis, so it is drawn as an open marker at the
# top of its interval instead. Every row keeps a range (zero-length for those),
# so points and ranges dodge together.
rate_points <- function(d) {
  mutate(d,
         shown    = ifelse(indels > 0, rate, high),
         bar_low  = ifelse(indels > 0, low, high),
         estimate = ifelse(indels > 0, "InDels observed", "none, upper bound"))
}

rate_layers <- function(dodge_width = 0.6) {
  dodge <- position_dodge(width = dodge_width)
  list(
    geom_linerange(aes(ymin = bar_low, ymax = high), position = dodge, linewidth = 0.5),
    geom_point(aes(shape = estimate), position = dodge, size = 2.2),
    scale_shape_manual(values = c("InDels observed" = 16, "none, upper bound" = 1), name = NULL),
    scale_y_log10()
  )
}

# Read positions at each distance from the nearest read end, over a set of
# reads, given the bin edges. A read of length L has two positions at every
# distance below L / 2 and none beyond, so a distance bin is only covered by the
# reads long enough to reach it. Without this the count of calls per distance
# falls away with distance for that reason alone.
availability <- function(read_length, breaks) {
  half     <- read_length / 2
  n_bins   <- length(breaks) - 1L
  width    <- diff(breaks)
  left     <- breaks[-length(breaks)]
  stops_in <- findInterval(half, breaks)   # the bin where the read runs out
  inside   <- stops_in >= 1L & stops_in <= n_bins
  bin      <- factor(stops_in[inside], levels = seq_len(n_bins))
  stopped  <- as.numeric(table(bin))                          # reads ending in the bin
  reach    <- as.numeric(tapply(half[inside], bin, sum))      # how far those reads reach
  reach[is.na(reach)] <- 0
  # A read reaching past the bin covers all of it; one that stops inside covers
  # only the part up to half its length. Two positions per distance, one per end.
  2 * (width * (length(half) - cumsum(stopped)) + (reach - left * stopped))
}

# Smallest rotation of a sequence or of its reverse complement, the form
# canonicalise() in scripts/parse_trf.py gives the tract motifs.
rotations <- function(s) {
  n <- nchar(s)
  substring(strrep(s, 2), seq_len(n), seq_len(n) + n - 1)
}

canonicalise <- function(seqs) {
  vapply(seqs, function(s) {
    rc <- paste(rev(strsplit(chartr("ACGT", "TGCA", s), "")[[1]]), collapse = "")
    min(rotations(s), rotations(rc))
  }, character(1), USE.NAMES = FALSE)
}

# Sample groups, straight from config.yaml. A sample may be in several lists.
config <- yaml::read_yaml(config_path)
group_keys <- function(key) {
  entries <- config[[key]]
  if (is.null(entries)) return(character(0))
  unlist(lapply(names(entries), function(sp) paste(sp, entries[[sp]], sep = "/")),
         use.names = FALSE)
}
tissue_keys      <- group_keys("tissue_samples")
methylation_keys <- group_keys("methylation_samples")
sperm_keys       <- group_keys("sperm_samples")

# Age of the individual in years, from config.yaml, ready to join on species and
# sample. Samples with no age on the sample sheet are simply absent from it.
sample_ages <- {
  entries <- config[["sample_age"]]
  bind_rows(lapply(names(entries), function(sp)
    tibble(species = sp,
           sample  = names(entries[[sp]]),
           age     = as.numeric(unlist(entries[[sp]], use.names = FALSE)))))
}

cat("config lists:", length(tissue_keys), "tissue,", length(methylation_keys),
    "germline/somatic,", length(sperm_keys), "sperm sample(s);",
    nrow(sample_ages), "samples with an age\n")

## What is in the results folder

In [ ]:
indel_files <- list.files(results_dir, pattern = "^indels\\.tsv$",
                          recursive = TRUE, full.names = TRUE)
if (length(indel_files) == 0) stop("No indels.tsv under ", results_dir)

runs <- tibble(dir = dirname(indel_files)) |>
  mutate(
    parts   = strsplit(sub(paste0("^", results_dir, "/"), "", dir), "/"),
    species = vapply(parts, `[`, character(1), 1),
    sample  = vapply(parts, `[`, character(1), 2)
  ) |>
  select(species, sample, dir) |>
  mutate(
    key       = paste(species, sample, sep = "/"),
    label     = paste(species, sample, sep = " / "),
    is_tissue = key %in% tissue_keys,
    is_methyl = key %in% methylation_keys,
    is_sperm  = key %in% sperm_keys,
    group     = case_when(is_tissue ~ "tissue",
                          is_methyl ~ "germline/somatic",
                          is_sperm  ~ "sperm",
                          TRUE      ~ "other"),
    # Not every sample has every file: a sample can have no calls at all, and a
    # copy from the cluster can be incomplete.
    has_calls         = has_rows(file.path(dir, "indels.tsv")),
    has_opportunities = has_rows(file.path(dir, "opportunities.tsv")),
    has_report        = has_rows(file.path(dir, "filter_report.tsv")),
    has_classified    = has_rows(file.path(dir, "indels_classified.tsv"))
  ) |>
  left_join(sample_ages, by = c("species", "sample"))
if (!is.null(species_keep)) runs <- filter(runs, species %in% species_keep)
if (nrow(runs) == 0) stop("No samples left under ", results_dir)

cat(nrow(runs), "samples:\n")
runs |> count(group, name = "samples") |> print()

runs |>
  select(species, sample, group, age, is_tissue, is_methyl, is_sperm,
         has_calls, has_opportunities, has_classified) |>
  arrange(group, species, sample) |>
  print(n = 60)

# What each gap means for the sections below.
if (any(!runs$has_calls)) {
  cat("\nno calls (the file is there but empty), so absent from every count:\n  ",
      paste(runs$label[!runs$has_calls], collapse = ", "), "\n")
}
if (any(!runs$has_opportunities)) {
  cat("\nno opportunities.tsv, so these samples can be counted but not turned",
      "into a rate:\n  ",
      paste(runs$label[!runs$has_opportunities], collapse = ", "), "\n")
}

# Samples named in a config list that have no results directory at all.
if (any(is.na(runs$age))) {
  cat("\nno age in config.yaml:\n  ",
      paste(runs$label[is.na(runs$age)], collapse = ", "), "\n")
}

missing <- setdiff(c(tissue_keys, methylation_keys, sperm_keys), runs$key)
if (length(missing) > 0) {
  cat("\nin a config list but no results under", results_dir, ":\n  ",
      paste(missing, collapse = ", "), "\n")
}

In [ ]:
# Rows each final filter removed from the calls. Only the kmerQ threshold is
# applied to the opportunities as well.
runs |>
  filter(has_report) |>
  mutate(report = lapply(file.path(dir, "filter_report.tsv"), read_tsv,
                         show_col_types = FALSE, progress = FALSE)) |>
  select(species, sample, group, report) |>
  unnest(report) |>
  mutate(share_kept = round(rows_after / rows_before, 4)) |>
  print(n = 60)

In [ ]:
if (!any(runs$has_calls)) stop("None of the samples has any calls")

indels_raw <- file.path(runs$dir[runs$has_calls], "indels.tsv") |>
  lapply(read_calls) |>
  bind_rows()

n_calls <- nrow(indels_raw)

# Every call of every sample. `unique` is kept as a column throughout, so no
# call is dropped here for being non-unique.
indels <- indels_raw |>
  filter(is.na(distance_to_indel_in_hap) | distance_to_indel_in_hap <= max_hap_distance,
         ToreadEnd >= min_read_end_distance) |>
  mutate(
    size      = nchar(indel_string),
    label     = paste(species, sample, sep = " / "),
    unique    = factor(unique, levels = c("unique", "non-unique")),
    # Contigs of a hifiasm diploid assembly are named h1tg... and h2tg...
    haplotype = ifelse(grepl("^h[12]tg", contig), substr(contig, 1, 2), "other")
  ) |>
  left_join(select(runs, species, sample, group, is_tissue, is_methyl, is_sperm),
            by = c("species", "sample")) |>
  group_by(species, sample, contig, position, indel_type, indel_string) |>
  mutate(reads_at_site = n()) |>
  ungroup()

cat("InDels:", nrow(indels), "of", n_calls, "kept with max_hap_distance =", max_hap_distance,
    "and min_read_end_distance =", min_read_end_distance, "\n")
cat("samples with calls:", n_distinct(indels$label), "\n")

# The opportunities as read, and summed per context. The calls are not filtered
# on mapQ, so the denominator is not either. They carry no unique flag: one set
# of read positions is the denominator for both classes of call.
rateable <- filter(runs, has_opportunities)
if (nrow(rateable) == 0) stop("No opportunities.tsv under ", results_dir)

opportunities_raw <- file.path(rateable$dir, "opportunities.tsv") |>
  lapply(read_tsv, show_col_types = FALSE, progress = FALSE,
         col_types = cols(tissue = col_character())) |>
  bind_rows() |>
  mutate(size = nchar(context))

opportunities <- opportunities_raw |>
  group_by(species, sample, size, context) |>
  summarise(opportunities = sum(count), .groups = "drop")

# Only the samples that have both go into a rate. Every rate table below is
# built from `opportunities`, so they are all restricted to these samples.
cat("samples with a rate:", n_distinct(paste(opportunities$species, opportunities$sample)), "\n")

# Every InDel sequence should be one of the counted contexts.
no_opportunities <- indels |>
  semi_join(rateable, by = c("species", "sample")) |>
  count(species, sample, size, canonical_indel) |>
  anti_join(opportunities, by = c("species", "sample", "size", "canonical_indel" = "context"))
if (nrow(no_opportunities) > 0) {
  warning(nrow(no_opportunities), " InDel sequence(s) have no opportunities")
}

opportunities |>
  group_by(species, sample, size) |>
  summarise(contexts = n(), opportunities = sum(opportunities), .groups = "drop") |>
  print(n = 60)

## Unique and non-unique calls

The first split. `unique` is set during the pileup, before any filter: a
position is `non-unique` when more than one read there carries an InDel of any
size. A true new mutation sits on one read, so non-unique calls are mostly
assembly errors or unresolved variants.

`reads_at_site` counts, among the calls kept here, how many reads carry the same
InDel at the same position. The two disagree when reads at a position carry
InDels of different sizes.

Both classes share the opportunities as denominator, so the two rates say how
often each kind of call arises per read position.

In [ ]:
unique_split <- indels |>
  count(label, group, size, unique, name = "indels") |>
  group_by(label, size) |>
  mutate(share = round(indels / sum(indels), 3)) |>
  ungroup()

print(unique_split, n = 60)

unique_rates <- opportunities |>
  group_by(species, sample, size) |>
  summarise(opportunities = sum(opportunities), .groups = "drop") |>
  cross_join(tibble(unique = factor(c("unique", "non-unique"),
                                    levels = c("unique", "non-unique")))) |>
  left_join(count(indels, species, sample, size, unique, name = "indels"),
            by = c("species", "sample", "size", "unique")) |>
  mutate(indels = coalesce(indels, 0L), label = paste(species, sample, sep = " / ")) |>
  left_join(select(runs, species, sample, group), by = c("species", "sample")) |>
  add_rate()

options(repr.plot.width = 12, repr.plot.height = facet_height(n_distinct(unique_rates$label)))
print(unique_rates |>
  rate_points() |>
  ggplot(aes(x = factor(size), y = shown, colour = unique, group = unique)) +
  rate_layers() +
  scale_colour_manual(values = unique_colours) +
  facet_wrap(~ label, ncol = 4) +
  labs(title = "Unique and non-unique calls",
       x = "InDel size (bp)", y = "calls per read position", colour = NULL) +
  theme_minimal())

# Reads carrying the same InDel at the same position, among the calls kept here.
indels |>
  distinct(label, contig, position, indel_type, indel_string, reads_at_site) |>
  mutate(reads = ifelse(reads_at_site >= 5, "5+", as.character(reads_at_site))) |>
  count(label, reads) |>
  group_by(label) |>
  mutate(share = round(n / sum(n), 3)) |>
  ungroup() |>
  pivot_wider(names_from = reads, values_from = c(n, share), values_fill = 0) |>
  print(n = 60, width = Inf)
options(repr.plot.width = 11, repr.plot.height = 6)

## Rate per InDel size

All samples. For a size, the opportunities are every read position counted under
that size, so insertions and deletions of the same size share one denominator.

In [ ]:
size_rates <- opportunities |>
  group_by(species, sample, size) |>
  summarise(opportunities = sum(opportunities), .groups = "drop") |>
  cross_join(expand_grid(indel_type = c("deletion", "insertion"),
                         unique = factor(c("unique", "non-unique"),
                                         levels = c("unique", "non-unique")))) |>
  left_join(count(indels, species, sample, size, indel_type, unique, name = "indels"),
            by = c("species", "sample", "size", "indel_type", "unique")) |>
  mutate(indels = coalesce(indels, 0L), label = paste(species, sample, sep = " / ")) |>
  left_join(select(runs, species, sample, group), by = c("species", "sample")) |>
  add_rate()

size_rates |>
  select(label, group, size, indel_type, unique, indels, opportunities, rate) |>
  arrange(label, size, indel_type, unique) |>
  print(n = 60)

options(repr.plot.width = 13, repr.plot.height = facet_height(n_distinct(size_rates$label), per_row = 4))
size_rates |>
  rate_points() |>
  ggplot(aes(x = factor(size), y = shown, colour = indel_type, group = interaction(indel_type, unique))) +
  rate_layers() +
  scale_colour_manual(values = type_colours) +
  facet_grid(label ~ unique) +
  labs(title = "InDel rate by size",
       x = "InDel size (bp)", y = "InDels per read position", colour = NULL) +
  theme_minimal() +
  theme(strip.text.y = element_text(angle = 0, hjust = 0))
options(repr.plot.width = 11, repr.plot.height = 6)

## Rate per context

The InDel spectrum as a rate: each canonical sequence divided by the read
positions where it could have been inserted or deleted. 1 bp contexts are only
A and C, since a base and its complement share one label. Unique calls only in
the plot; the table keeps both classes.

In [ ]:
context_rates <- opportunities |>
  cross_join(expand_grid(indel_type = c("deletion", "insertion"),
                         unique = factor(c("unique", "non-unique"),
                                         levels = c("unique", "non-unique")))) |>
  left_join(count(indels, species, sample, size, canonical_indel, indel_type, unique,
                  name = "indels"),
            by = c("species", "sample", "size", "context" = "canonical_indel",
                   "indel_type", "unique")) |>
  mutate(indels = coalesce(indels, 0L), label = paste(species, sample, sep = " / ")) |>
  add_rate()

# The contexts carrying the most InDels, both classes.
context_rates |>
  group_by(context, size, unique) |>
  summarise(indels = sum(indels), .groups = "drop") |>
  pivot_wider(names_from = unique, values_from = indels, values_fill = 0) |>
  arrange(desc(unique)) |>
  print(n = 20)

options(repr.plot.width = 14, repr.plot.height = facet_height(n_distinct(context_rates$label), per_row = 1, row_height = 1.6))
context_rates |>
  filter(unique == "unique") |>
  rate_points() |>
  ggplot(aes(x = reorder(context, -rate, max), y = shown,
             colour = indel_type, group = indel_type)) +
  rate_layers() +
  scale_colour_manual(values = type_colours) +
  facet_grid(label ~ size, scales = "free_x", space = "free_x",
             labeller = labeller(size = function(x) paste(x, "bp"))) +
  labs(title = "InDel rate by context",
       x = "context (smaller of the sequence and its reverse complement)",
       y = "InDels per read position", colour = NULL) +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1),
        strip.text.y = element_text(angle = 0, hjust = 0))
options(repr.plot.width = 11, repr.plot.height = 6)

## Where the calls are

The rate per contig against the contig's opportunities. With one rate for the
whole genome, contigs scatter inside the funnel (99.9% Poisson range around the
sample's rate). Contigs far above it carry more calls than chance allows, which
points to local problems such as assembly errors or collapsed duplications
rather than mutation. Non-unique calls are where those problems show first, so
the two classes are drawn separately.

In [ ]:
contig_rates <- opportunities_raw |>
  group_by(species, sample, contig) |>
  summarise(opportunities = sum(count), .groups = "drop") |>
  cross_join(tibble(unique = factor(c("unique", "non-unique"),
                                    levels = c("unique", "non-unique")))) |>
  left_join(count(indels, species, sample, contig, unique, name = "indels"),
            by = c("species", "sample", "contig", "unique")) |>
  mutate(indels    = coalesce(indels, 0L),
         label     = paste(species, sample, sep = " / "),
         haplotype = ifelse(grepl("^h[12]tg", contig), substr(contig, 1, 2), "other")) |>
  add_rate() |>
  group_by(species, sample, unique) |>
  mutate(sample_rate = sum(indels) / sum(opportunities),
         expected    = sample_rate * opportunities,
         p_excess    = ppois(indels - 1, expected, lower.tail = FALSE),
         flagged     = p_excess < 0.001 / n()) |>
  ungroup()

# A contig with no calls has rate 0 and cannot go on a log axis, so the plot
# shows the contigs that carry at least one call; the tables below count them all.
options(repr.plot.width = 13, repr.plot.height = facet_height(n_distinct(contig_rates$label), per_row = 3))
print(ggplot(filter(contig_rates, indels > 0), aes(x = opportunities, y = rate, colour = unique)) +
  geom_point(alpha = 0.6, size = 1.4) +
  geom_hline(aes(yintercept = sample_rate, colour = unique), linetype = "dashed") +
  scale_x_log10() +
  scale_y_log10() +
  scale_colour_manual(values = unique_colours) +
  facet_wrap(~ label, ncol = 3) +
  labs(title = "InDel rate by contig",
       x = "opportunities on the contig", y = "calls per read position", colour = NULL) +
  theme_minimal())
options(repr.plot.width = 11, repr.plot.height = 6)

cat("contigs with a significant excess of unique calls:\n")
contig_rates |>
  filter(flagged, unique == "unique") |>
  arrange(p_excess) |>
  select(label, contig, indels, expected, rate, sample_rate, p_excess) |>
  print(n = 30)

In [ ]:
# The two haplotypes of the diploid assembly separately. They should give the
# same rate; a difference points to one haplotype being assembled worse.
contig_rates |>
  group_by(label, unique, haplotype) |>
  summarise(contigs = n(), indels = sum(indels), opportunities = sum(opportunities),
            .groups = "drop") |>
  add_rate() |>
  select(label, unique, haplotype, contigs, indels, rate) |>
  pivot_wider(names_from = haplotype, values_from = c(contigs, indels, rate)) |>
  print(n = 60, width = Inf)

## Distance to the read end

HiFi errors concentrate at read ends, so calls crowding the ends are a sign of
calls made from error-prone sequence rather than mutation.

Counts alone cannot show this. A read of length L has two positions at every
distance below L / 2 and none beyond it, so only the reads longer than 2d reach
distance d. At HiFi read lengths that only bites past a few kb: nearer than
that every read reaches, so the count expected under a flat rate is constant,
and further out it falls as the reads run out. The first plot draws the observed
counts against exactly that expectation. The
second divides the counts by the positions available at each distance, which is
the profile to read for an end effect: flat means calls fall anywhere along a
read, a rise towards 0 means they crowd the ends.

That denominator is read geometry over the reads that carry a call, not the
filtered opportunity table used elsewhere in this notebook. Two details of it:
`read_length` is the length of the read sequence, while `ToreadEnd` is measured
on the stretch of reference the read aligns to, so on a soft-clipped read the
denominator is a little generous at the far end of the span; and a read counts
once however many calls it carries. So the profile
answers where along a read the calls sit, and its height is not the sample's
mutation rate. `min_read_end_distance` in the setup cell drops the near end
altogether; the repeat calls use 100 bp.

In [ ]:
end_distances <- c(25, 50, 100, 250, 500, 1000)

indels |>
  group_by(label, unique) |>
  reframe(
    distance = end_distances,
    observed = vapply(end_distances, function(d) mean(ToreadEnd < d), numeric(1)),
    expected = vapply(end_distances, function(d) mean(pmin(1, 2 * d / read_length)), numeric(1))
  ) |>
  mutate(excess = round(observed / expected, 2)) |>
  select(label, unique, distance, excess) |>
  pivot_wider(names_from = distance, values_from = excess, names_prefix = "<") |>
  print(n = 60, width = Inf)

bin_width <- 100

# The far tail rests on the few longest reads, so a bin is shown once enough
# positions reach it.
min_positions_per_bin <- 1e5

breaks <- seq(0, ceiling(max(indels$ToreadEnd) / bin_width) * bin_width, by = bin_width)

# One row per read, so a read carrying several calls counts once in the
# denominator, and the two classes of call share it as they do elsewhere.
positions_per_bin <- indels |>
  distinct(label, readID, read_length) |>
  group_by(label) |>
  reframe(bin           = seq_len(length(breaks) - 1L),
          opportunities = availability(read_length, breaks))

read_end_profile <- positions_per_bin |>
  cross_join(tibble(unique = factor(c("unique", "non-unique"),
                                    levels = c("unique", "non-unique")))) |>
  left_join(indels |>
              mutate(bin = findInterval(ToreadEnd, breaks)) |>
              count(label, unique, bin, name = "indels"),
            by = c("label", "unique", "bin")) |>
  mutate(indels   = coalesce(indels, 0L),
         distance = breaks[bin] + bin_width / 2) |>
  filter(opportunities >= min_positions_per_bin) |>
  group_by(label, unique) |>
  mutate(overall  = sum(indels) / sum(opportunities),
         expected = overall * opportunities) |>
  ungroup() |>
  add_rate() #|>
  # filter(label %in% c("gibbon / pink_eric", "marmoset / sidonia", "bonobo / matos"))

cat("profile over", n_distinct(read_end_profile$bin), "bins of", bin_width, "bp, reaching",
    format(max(read_end_profile$distance), big.mark = ","), "bp;",
    sum(read_end_profile$indels), "of", nrow(indels), "calls are in a shown bin\n")

# Counts over the full span, against the counts expected if calls were spread
# evenly along the reads: the sample's average rate times the positions in the
# bin. The dashed line is flat while every read still reaches that far and falls
# only where the reads run out, so near the end it is a like-for-like comparison.
# It holds the same total as the observed curve, being that total spread evenly,
# so an excess at the end is paid for by the rest of the span sitting below it.
options(repr.plot.width = 13, repr.plot.height = facet_height(n_distinct(read_end_profile$label), per_row = 3))
p_read_end_counts <- (ggplot(read_end_profile, aes(x = distance)) +
  geom_line(aes(y = expected, group = unique), colour = "grey55", linetype = "dashed") +
  geom_line(aes(y = indels, colour = unique)) +
  scale_colour_manual(values = unique_colours) +
  facet_wrap(~ label, ncol = 3, scales = "free_y") +
  labs(title = "Calls by distance to the read end",
       x = "distance to read end (bp)", y = "calls in the bin", colour = NULL) +
  theme_minimal()) +
  theme(plot.title.position = "plot",                                  # centre over the whole figure, not the panels
        plot.title    = element_text(hjust = 0.5),
        plot.caption  = element_text(hjust = 0.5, colour = "grey40"))
print(p_read_end_counts)

# The same profile per available position. Log x, so the near end where the
# errors sit is legible next to the full span.
p_read_end_rate  <- (ggplot(read_end_profile, aes(x = distance, y = rate, colour = unique, fill = unique)) +
  geom_ribbon(aes(ymin = low, ymax = high), alpha = 0.2, colour = NA) +
  geom_line() +
  geom_hline(aes(yintercept = overall, colour = unique), linetype = "dashed") +
  scale_x_log10() +
  scale_colour_manual(values = unique_colours) +
  scale_fill_manual(values = unique_colours, guide = "none") +
  facet_wrap(~ label, ncol = 3, scales = "free_y") +
  labs(title = "Rate by distance to the read end",
       x = "distance to read end (bp, log scale)", y = "calls per read position", colour = NULL) +
  theme_minimal()) + 
  theme(plot.title.position = "plot",                                 
        plot.title    = element_text(hjust = 0.5),
        plot.caption  = element_text(hjust = 0.5, colour = "grey40"))
print(p_read_end_rate)
options(repr.plot.width = 11, repr.plot.height = 6)

### The same profile for a slide

One panel per sample answers "which sample", which is what the plots above are
for. For a slide the question is "does the effect exist", so every sample goes on
one pair of axes and the panels carry the comparison that matters: unique calls
against non-unique ones.

Each bin's rate is divided by that sample's own average along the read, so
samples whose absolute rates differ by an order of magnitude still lie on one
axis, and the y axis reads directly as "times this sample's own average". A flat
bundle at 1 means position along the read carries no information; the unique
panel does not stay flat.

`save_svg()` writes any plot in this notebook to `plots/` as SVG, which
PowerPoint opens with the text still editable. Pass any of the other plot objects
to it the same way, for instance `save_svg(p_read_end_rate, "read_end_rate.svg",
height = 14)`.

In [ ]:
# The profile is flat well before the reads run out, and the last few kb rest on
# a handful of long reads per bin, which is noise on a slide. The full span is in
# the plots above; this one stops where the lines are still worth reading.
slide_max_distance <- 100000

# Bins with no calls have no ratio to draw on a log axis, so they are left out;
# a sample's line simply stops where its calls do.
read_end_slide <- read_end_profile |>
  filter(indels > 0, distance <= slide_max_distance) |>
  left_join(distinct(runs, label, species), by = "label") |>
  mutate(enrichment = rate / overall)

# The median across samples, for the eye to follow, where at least half of them
# still reach.
read_end_mean <- read_end_slide |>
  group_by(unique, distance) |>
  summarise(enrichment = mean(enrichment), samples = n(), .groups = "drop") |>
  filter(samples >= 0.5 * n_distinct(read_end_slide$label))

p_read_end_slide <- ggplot(read_end_slide, aes(x = distance, y = enrichment)) +
  geom_hline(yintercept = 1, colour = "grey30", linewidth = 0.4) +
  geom_line(aes(group = label, colour = species), linewidth = 0.3, alpha = 0.6) +
  geom_line(data = read_end_mean, linewidth = 1.3, colour = "grey10") +
  scale_x_log10(breaks = c(100, 300, 1000, 3000, 5000, 10000),
                labels = c("100", "300", "1,000", "3,000", "5,000", "100000")) +
  scale_y_log10(breaks = c(0.5, 1, 2, 5, 10, 20)) +
  scale_colour_discrete(guide = guide_legend(nrow = 2, override.aes = list(linewidth = 1.2, alpha = 1))) +
  facet_wrap(~ unique, labeller = labeller(unique = c(unique = "unique calls",
                                                      `non-unique` = "non-unique calls"))) +
  labs(title = "Distance to the read end",
       x = "distance to the nearest read end (bp)",
       y = "times the sample's average rate",
       colour = NULL) +
  theme_minimal(base_size = 14) +
  theme(legend.position = "bottom",
        panel.grid.minor = element_blank(),
        plot.title    = element_text(hjust = 0.5),
        strip.text = element_text(face = "bold", size = 14),
        plot.caption = element_text(colour = "grey40"))

options(repr.plot.width = 12, repr.plot.height = 6.5)
print(p_read_end_slide)
options(repr.plot.width = 11, repr.plot.height = 6)

save_svg(p_read_end_slide, "read_end_profile.svg")

## Haploid check

`haploid_check` is TRUE when, in the read's alignment to the haploid assembly,
the first InDel of the same type at or after the call's read position has the
same size. How far away that InDel is (`distance_to_indel_in_hap`) is not
limited, so a call can pass on an InDel several kb away. `max_hap_distance` in
the setup cell requires the two to be close.

In [ ]:
indels |>
  mutate(distance = cut(distance_to_indel_in_hap, c(-Inf, 0, 10, 100, 1000, Inf),
                        labels = c("0", "1-10", "11-100", "101-1000", ">1000"))) |>
  count(label, unique, distance) |>
  group_by(label, unique) |>
  mutate(share = round(n / sum(n), 3)) |>
  ungroup() |>
  select(-n) |>
  pivot_wider(names_from = distance, values_from = share, values_fill = 0) |>
  print(n = 60, width = Inf)

# Other read-level signs of artefacts: a strand imbalance, soft clipping in the
# haploid alignment, and the number of passes behind the consensus read (np).
indels |>
  group_by(label, unique) |>
  summarise(calls            = n(),
            plus_strand      = round(mean(diploid_direction == "+"), 3),
            haploid_softclip = round(mean(softclip_haploid), 3),
            median_passes    = median(np, na.rm = TRUE),
            under_5_passes   = round(mean(np < 5, na.rm = TRUE), 3),
            .groups = "drop") |>
  print(n = 60, width = Inf)

# Group: tissue samples

Samples listed under `tissue_samples` in config.yaml. Their reads carry a tissue
label, and so do their opportunities, so each tissue's calls are divided by that
tissue's own read positions: coverage differences between tissues cancel out.
Reads without a label in these samples count as NA.

In [ ]:
tissue_runs <- filter(runs, is_tissue)

if (nrow(tissue_runs) == 0) {
  message("No sample under tissue_samples in config.yaml has results here.")
} else {
  cat("tissue samples:", paste(tissue_runs$label, collapse = ", "), "\n")
  if (any(!tissue_runs$has_opportunities)) {
    cat("  no opportunities, so absent from the rates below:",
        paste(tissue_runs$label[!tissue_runs$has_opportunities], collapse = ", "), "\n")
  }

  tissue_rates <- opportunities_raw |>
    semi_join(tissue_runs, by = c("species", "sample")) |>
    group_by(species, sample, size, tissue) |>
    summarise(opportunities = sum(count), .groups = "drop") |>
    cross_join(tibble(unique = factor(c("unique", "non-unique"),
                                      levels = c("unique", "non-unique")))) |>
    left_join(count(semi_join(indels, tissue_runs, by = c("species", "sample")),
                    species, sample, size, tissue, unique, name = "indels"),
              by = c("species", "sample", "size", "tissue", "unique")) |>
    mutate(indels = coalesce(indels, 0L),
           tissue = ifelse(is.na(tissue), "NA (unlabelled)", tissue),
           label  = paste(species, sample, sep = " / ")) |>
    add_rate()

  tissue_rates |>
    group_by(label, tissue, unique) |>
    summarise(indels = sum(indels), opportunities = first(opportunities), .groups = "drop") |>
    print(n = 60)

  options(repr.plot.width = 13, repr.plot.height = facet_height(n_distinct(tissue_rates$label), per_row = 2))
  print(tissue_rates |>
    rate_points() |>
    ggplot(aes(x = tissue, y = shown, colour = unique, group = unique)) +
    rate_layers() +
    scale_colour_manual(values = unique_colours) +
    facet_grid(label ~ size, labeller = labeller(size = function(x) paste(x, "bp"))) +
    labs(title = "InDel rate by tissue",
         x = NULL, y = "InDels per read position", colour = NULL) +
    theme_minimal() +
    theme(axis.text.x = element_text(angle = 45, hjust = 1),
          strip.text.y = element_text(angle = 0, hjust = 0)))
  options(repr.plot.width = 11, repr.plot.height = 6)
}

### Rate per tissue, read ends removed

Only the samples with a tissue label, only unique calls, and only calls further
than 300 bp from a read end, where the profile above shows the near-end excess
has died away.

Cutting the numerator at 300 bp means cutting the denominator too, or the rate
is deflated by whatever share of read positions lies inside the cut. The
opportunity table has no distance column, so that share is estimated from the
reads themselves: a read of length L holds `min(2 * 300, L)` positions within
300 bp of one of its ends, so the tissue's positions are scaled by the share
outside. It is estimated per sample and tissue from the reads carrying a call
there, which are a little longer than average, so the correction is a slight
underestimate — about 4% of positions on a 15 kb read, and near enough equal
across the tissues of one sample that the comparison between them is unaffected.

Reads with no tissue label are left out, so the rows are tissues, not a
catch-all.

In [ ]:
read_end_cut <- 300   # calls at or below this distance from a read end are dropped

tissue_runs <- filter(runs, is_tissue, has_opportunities)

if (nrow(tissue_runs) == 0) {
  message("No tissue sample has opportunities here.")
} else {
  tissue_calls <- indels |>
    semi_join(tissue_runs, by = c("species", "sample")) |>
    filter(unique == "unique", ToreadEnd > read_end_cut, !is.na(tissue))

  # The share of a tissue's read positions left after the cut. A read of length
  # L holds min(2 * cut, L) positions within `cut` of an end.
  usable <- indels |>
    semi_join(tissue_runs, by = c("species", "sample")) |>
    filter(!is.na(tissue)) |>
    distinct(species, sample, tissue, readID, read_length) |>
    group_by(species, sample, tissue) |>
    summarise(usable_share = 1 - sum(pmin(2 * read_end_cut, read_length)) / sum(read_length),
              reads = n(), .groups = "drop")

  tissue_rates_trim <- opportunities_raw |>
    semi_join(tissue_runs, by = c("species", "sample")) |>
    filter(!is.na(tissue)) |>
    group_by(species, sample, size, tissue) |>
    summarise(positions = sum(count), .groups = "drop") |>
    inner_join(usable, by = c("species", "sample", "tissue")) |>
    left_join(count(tissue_calls, species, sample, size, tissue, name = "indels"),
              by = c("species", "sample", "size", "tissue")) |>
    mutate(indels        = coalesce(indels, 0L),
           opportunities = positions * usable_share,
           label         = paste(species, sample, sep = " / ")) |>
    add_rate()

  cat(nrow(tissue_calls), "unique calls beyond", read_end_cut, "bp from a read end, in",
      n_distinct(tissue_rates_trim$label), "samples and",
      n_distinct(tissue_rates_trim$tissue), "tissues\n")
  cat("positions kept by the cut:",
      paste0(round(100 * range(tissue_rates_trim$usable_share), 1), "%", collapse = " to "), "\n")

  tissue_rates_trim |>
    select(label, tissue, size, indels, opportunities, rate) |>
    mutate(opportunities = signif(opportunities, 3), rate = signif(rate, 3)) |>
    arrange(label, tissue, size) |>
    print(n = 40)

  # Tissues ordered by their rate across samples, so the ranking is the y axis.
  dodge <- position_dodge(width = 0.7)
  p_tissue_rate <- tissue_rates_trim |>
    rate_points() |>
    mutate(tissue = reorder(tissue, rate, median)) |>
    ggplot(aes(x = shown, y = tissue, colour = species)) +
    geom_linerange(aes(xmin = bar_low, xmax = high, group = label),
                   position = dodge, linewidth = 0.4) +
    geom_point(aes(shape = estimate, group = label), position = dodge, size = 1.9) +
    scale_shape_manual(values = c("InDels observed" = 16, "none, upper bound" = 1), name = NULL) +
    scale_x_log10() +
    facet_wrap(~ size, nrow = 1, labeller = labeller(size = function(x) paste(x, "bp InDels"))) +
    labs(title = "InDel rate by tissue, nonrepetitive DNA",
         x = "rate", y = NULL, colour = NULL) +
    theme_minimal(base_size = 13) +
    theme(legend.position = "bottom",
          plot.title    = element_text(hjust = 0.5),
          panel.grid.minor = element_blank(),
          plot.title.position = "plot")

  options(repr.plot.width = 13, repr.plot.height = 7)
  print(p_tissue_rate)
  options(repr.plot.width = 11, repr.plot.height = 6)

  save_svg(p_tissue_rate, "tissue_rate.svg", width = 12, height = 6.5)
}

# Group: germline/somatic samples

Samples listed under `methylation_samples` in config.yaml. For these the
pipeline writes `indels_classified.tsv` next to `indels.tsv`: the same calls plus
five columns describing the read each call sits on.

| column | meaning |
|---|---|
| `max_germline` | best likelihood among the germline (`Sperm.*`) cell types |
| `max_somatic` | best likelihood among the somatic cell types |
| `classification_diff` | `max_germline - max_somatic`; positive looks germline |
| `classification_bin` | that difference cut into bins |
| `CpGs_onRead` | CpGs on the read, i.e. the evidence behind the likelihoods |

A difference of -1 or more counts as germline. The section starts with counts
and shares, and then turns them into rates with the approximate denominator
described below.

In [ ]:
methyl_runs <- filter(runs, is_methyl)
classified_paths <- file.path(methyl_runs$dir[methyl_runs$has_classified],
                              "indels_classified.tsv")

classified <- NULL
if (nrow(methyl_runs) == 0) {
  message("No sample under methylation_samples in config.yaml has results here.")
} else if (length(classified_paths) == 0) {
  message("No indels_classified.tsv for ", paste(methyl_runs$label, collapse = ", "),
          ". The classification step has to have run.")
} else {
  classification <- classified_paths |>
    lapply(read_calls) |>
    bind_rows() |>
    select(species, sample, readID, position, indel_string,
           max_germline, max_somatic, classification_diff, classification_bin, CpGs_onRead)

  # The filters the rest of the notebook applies are already in `indels`, so the
  # columns are joined onto it rather than read again.
  # A read that is not in the sample's likelihood table has no scores, so
  # classification_diff is NA there and `germline` is neither TRUE nor FALSE.
  classified <- indels |>
    inner_join(classification,
               by = c("species", "sample", "readID", "position", "indel_string")) |>
    mutate(germline = classification_diff >= -1,
           scored   = !is.na(classification_diff),
           classification_bin = factor(
             ifelse(scored, as.character(classification_bin), "not scored"),
             levels = c(as.character(sort(unique(classification_bin))), "not scored")))

  cat(nrow(classified), "calls joined to a read, in", n_distinct(classified$label),
      "sample(s);", sum(!classified$scored), "sit on a read that is not in the",
      "likelihood table\n")

  classified |>
    group_by(label, unique) |>
    summarise(calls          = n(),
              # summarise() evaluates in order, so `somatic` and the share use
              # the counts just computed, not the logical columns they came from.
              not_scored     = sum(!scored),
              scored         = calls - not_scored,
              germline       = sum(germline, na.rm = TRUE),
              somatic        = scored - germline,
              share_germline = round(germline / scored, 3),
              median_diff    = round(median(classification_diff, na.rm = TRUE), 2),
              .groups = "drop") |>
    print(n = 60)
}

In [ ]:
if (!is.null(classified)) {
  options(repr.plot.width = 12, repr.plot.height = facet_height(n_distinct(classified$label), per_row = 2))
  bin_counts <- count(classified, label, unique, classification_bin)
  print(ggplot(bin_counts, aes(x = classification_bin, y = n, fill = unique)) +
    geom_col(position = "dodge") +
    scale_fill_manual(values = unique_colours) +
    facet_wrap(~ label, ncol = 2, scales = "free_y") +
    labs(title = "Calls by classification bin",
         x = "classification_bin", y = "calls", fill = NULL) +
    theme_minimal())
  options(repr.plot.width = 11, repr.plot.height = 6)
}

### Rate per bin, on an approximate denominator

The calls carry a bin but `opportunities.tsv` does not: it counts read positions
with read identity aggregated away, so the denominator a rate per bin needs,

    P_b = the read positions contributed by the reads in bin b

is not in it. `classification_reads.tsv`, written by
`workflows/classification_denominator.py`, holds what is missing: how many of
the sample's reads fall in each bin and how many CpGs they carry, counted over
the whole likelihood table. The bins are then given a share of the sample's read
positions P:

    P_b  ~=  P * reads_b / reads_total          (flat)
    P_b  ~=  P * CpGs_b  / CpGs_total           (length-weighted)

Both split P exactly, so the bin rates average back to the sample's overall
rate, and the unit stays calls per read position as everywhere else here.

**What the flat split assumes.** That a read in one bin contributes as many
counted positions as a read in another. It is not quite true: a longer read
carries more CpGs, is classified more confidently and lands in a more extreme
bin, so the extreme bins hold longer reads and the flat split gives them too
small a denominator, overstating their rate. The length-weighted split uses CpGs
as a stand-in for read length and corrects most of that. Both are reported, and
the distance between them is the size of the effect.

The same filters as the tissue plot apply: unique calls only, and only calls
further than 300 bp from a read end, with the denominator scaled per bin by the
share of each bin's read positions that survive the cut. Calls on reads that are
not in the likelihood table have no bin and no denominator, so they are left
out; their share is in the table above.

The file is the same for both pipelines, so it is read from whichever result
tree has it.

In [ ]:
bin_rates <- NULL

if (is.null(classified)) {
  message("No classification loaded, nothing to put a denominator under.")
} else {
  # The counts are a property of the sample, so the file is the same in both
  # result trees; take whichever is there.
  counts_path <- function(species, sample) {
    candidates <- file.path(c(results_dir, repeat_indels_dir), species, sample,
                            "classification_reads.tsv")
    found <- candidates[file.exists(candidates)]
    if (length(found) == 0) NA_character_ else found[1]
  }

  read_counts <- methyl_runs |>
    mutate(path = mapply(counts_path, species, sample)) |>
    filter(!is.na(path)) |>
    mutate(counts = lapply(path, read_tsv, show_col_types = FALSE, progress = FALSE)) |>
    select(species, sample, label, counts) |>
    unnest(counts) |>
    group_by(species, sample) |>
    mutate(read_share = reads / sum(reads),
           cpg_share  = CpGs / sum(CpGs),
           germline   = classification_bin >= -1) |>
    ungroup()

  if (nrow(read_counts) == 0) {
    message("No classification_reads.tsv found. Run ",
            "gwf -f workflows/classification_denominator.py run")
  } else {
    cat(n_distinct(read_counts$label), "sample(s) with read counts,",
        format(sum(read_counts$reads), big.mark = ","), "reads in all\n")

    # How far the two splits disagree is how much read length depends on the bin.
    read_counts |>
      group_by(label) |>
      summarise(reads              = sum(reads),
                share_germline     = round(sum(read_share[germline]), 3),
                cpg_share_germline = round(sum(cpg_share[germline]), 3),
                .groups = "drop") |>
      print(n = 20)

    # The calls: unique, and clear of the read ends, as in the tissue plot.
    scored_calls <- classified |>
      filter(scored, unique == "unique", ToreadEnd > read_end_cut) |>
      mutate(classification_bin = as.numeric(as.character(classification_bin)))

    # The read-end cut removes positions from each bin too. Extreme bins hold
    # longer reads, so the share lost differs by bin and is estimated per bin.
    usable_bin <- scored_calls |>
      distinct(species, sample, classification_bin, readID, read_length) |>
      group_by(species, sample, classification_bin) |>
      summarise(usable_share = 1 - sum(pmin(2 * read_end_cut, read_length)) / sum(read_length),
                .groups = "drop")

    bin_rates <- opportunities |>
      semi_join(read_counts, by = c("species", "sample")) |>
      group_by(species, sample, size) |>
      summarise(positions = sum(opportunities), .groups = "drop") |>
      inner_join(select(read_counts, species, sample, label, classification_bin,
                        germline, read_share, cpg_share),
                 by = c("species", "sample"), relationship = "many-to-many") |>
      left_join(usable_bin, by = c("species", "sample", "classification_bin")) |>
      left_join(count(scored_calls, species, sample, size, classification_bin, name = "indels"),
                by = c("species", "sample", "size", "classification_bin")) |>
      mutate(indels        = coalesce(indels, 0L),
             usable_share  = coalesce(usable_share, 1),
             opportunities = positions * read_share * usable_share,
             cpg_weighted  = positions * cpg_share  * usable_share) |>
      add_rate() |>
      mutate(rate_cpg = indels / cpg_weighted)

    cat(nrow(scored_calls), "unique scored calls beyond", read_end_cut,
        "bp from a read end, over", n_distinct(bin_rates$label), "samples\n")

    # Germline against somatic, the bins pooled either side of -1.
    germline_rates <- bin_rates |>
      group_by(label, size, germline) |>
      summarise(indels = sum(indels), opportunities = sum(opportunities),
                cpg_weighted = sum(cpg_weighted), .groups = "drop") |>
      add_rate() |>
      mutate(rate_cpg = indels / cpg_weighted,
             cells    = ifelse(germline, "germline", "somatic"))

    germline_rates |>
      select(label, size, cells, indels, rate, rate_cpg) |>
      pivot_wider(names_from = cells, values_from = c(indels, rate, rate_cpg)) |>
      mutate(ratio     = round(rate_germline / rate_somatic, 2),
             ratio_cpg = round(rate_cpg_germline / rate_cpg_somatic, 2)) |>
      select(label, size, indels_germline, indels_somatic, ratio, ratio_cpg) |>
      print(n = 40)

    p_classification_rate <- bin_rates |>
      rate_points() |>
      ggplot(aes(x = classification_bin, y = shown, colour = species, group = label)) +
      geom_vline(xintercept = -1.5, colour = "grey60", linetype = "dashed") +
      geom_line(linewidth = 0.4, alpha = 0.8) +
      geom_point(aes(shape = estimate), size = 1.6) +
      scale_shape_manual(values = c("InDels observed" = 16, "none, upper bound" = 1), name = NULL) +
      scale_y_log10() +
      scale_x_continuous(breaks = sort(unique(bin_rates$classification_bin))) +
      facet_wrap(~ size, nrow = 1, labeller = labeller(size = function(x) paste(x, "bp InDels"))) +
      labs(title = "InDel rate by classification bin",
           x = "classification_bin      (negative: somatic, positive: germline)",
           y = "InDels per read position", colour = NULL) +
      theme_minimal(base_size = 13) +
      theme(legend.position = "bottom",
            panel.grid.minor = element_blank(),
            plot.title.position = "plot")

    options(repr.plot.width = 13, repr.plot.height = 6)
    print(p_classification_rate)
    options(repr.plot.width = 11, repr.plot.height = 6)

    save_svg(p_classification_rate, "classification_rate.svg", width = 12, height = 6)
  }
}

In [ ]:
if (!is.null(classified)) {
  # Only the scored reads have a difference to plot, and a read with 0 CpGs
  # cannot go on a log axis.
  shown <- filter(classified, scored, CpGs_onRead > 0)
  options(repr.plot.width = 12, repr.plot.height = facet_height(n_distinct(classified$label), per_row = 2))
  print(ggplot(shown, aes(x = CpGs_onRead, y = classification_diff, colour = unique)) +
    geom_point(alpha = 0.35, size = 0.7) +
    geom_hline(yintercept = -1, linetype = "dashed", colour = "grey40") +
    scale_x_log10() +
    scale_colour_manual(values = unique_colours) +
    facet_wrap(~ label, ncol = 2) +
    labs(title = "Evidence behind the classification",
         x = "CpGs on the read (log scale)", y = "classification_diff", colour = NULL) +
    theme_minimal())
  options(repr.plot.width = 11, repr.plot.height = 6)
}

if (!is.null(classified)) {
  both <- filter(classified, !is.na(tissue))
  if (nrow(both) == 0) {
    message("No sample has both a tissue label and a read classification: ",
            "tissue_samples and methylation_samples in config.yaml do not overlap.")
  } else {
    both |>
      filter(scored) |>
      count(label, tissue, unique, germline) |>
      pivot_wider(names_from = germline, values_from = n, values_fill = 0,
                  names_prefix = "germline_") |>
      print(n = 60)
  }
}

# Group: sperm samples

Samples listed under `sperm_samples` in config.yaml. Their reads carry no
per-read label: the whole sample is one tissue, so there is nothing to split on
within a sample. What is worth looking at is how they compare with each other
and with the other groups.

In [ ]:
sperm_runs <- filter(runs, is_sperm)

if (nrow(sperm_runs) == 0) {
  message("No sample under sperm_samples in config.yaml has results here.")
} else {
  cat("sperm samples:", paste(sperm_runs$label, collapse = ", "), "\n")
  if (any(!sperm_runs$has_opportunities)) {
    cat("  no opportunities, so absent from the rates below:",
        paste(sperm_runs$label[!sperm_runs$has_opportunities], collapse = ", "), "\n")
  }

  options(repr.plot.width = 12, repr.plot.height = 6)
  print(size_rates |>
    semi_join(sperm_runs, by = c("species", "sample")) |>
    filter(unique == "unique") |>
    rate_points() |>
    ggplot(aes(x = label, y = shown, colour = indel_type, group = indel_type)) +
    rate_layers() +
    scale_colour_manual(values = type_colours) +
    facet_wrap(~ size, labeller = labeller(size = function(x) paste(x, "bp"))) +
    labs(title = "InDel rate in sperm samples",
 x = NULL,
         y = "InDels per read position", colour = NULL) +
    theme_minimal() +
    theme(axis.text.x = element_text(angle = 45, hjust = 1)))
}

### Sperm against age

Every sperm read in the data, against the age of the animal it came from. Two
sources, kept apart by the marker: the samples under `sperm_samples` in
config.yaml, where the whole sample is sperm, and the sperm-labelled reads
inside the multi-tissue samples, where the denominator is that tissue's own
read positions.

Same filters as the other rate plots: unique calls, further than 300 bp from a
read end, with the denominator scaled by the share of read positions surviving
that cut.

The line is a fit through the human sperm series alone, which is the only
within-species age range here (24 to 49 years); the other points are single
animals of different species and are there for context, not for the trend. The
fit is a quasi-Poisson regression of the counts with the read positions as
offset, so it is a rate ratio per year of age and its dispersion is estimated
from the data rather than assumed to be Poisson.

In [ ]:
sperm_source <- runs |>
  filter(has_opportunities, is_sperm | is_tissue) |>
  mutate(from = ifelse(is_sperm, "whole sample", "sperm reads")) |>
  select(species, sample, label, from, age)

# A whole-sample sperm run has no tissue column to filter on; a multi-tissue
# sample contributes only the reads labelled sperm.
is_sperm_row <- function(d) filter(d, from == "whole sample" | tissue == "sperm")

sperm_calls <- indels |>
  inner_join(sperm_source, by = c("species", "sample")) |>
  is_sperm_row() |>
  filter(unique == "unique", ToreadEnd > read_end_cut)

usable <- indels |>
  inner_join(sperm_source, by = c("species", "sample")) |>
  is_sperm_row() |>
  distinct(species, sample, readID, read_length) |>
  group_by(species, sample) |>
  summarise(usable_share = 1 - sum(pmin(2 * read_end_cut, read_length)) / sum(read_length),
            .groups = "drop")

sperm_rates <- opportunities_raw |>
  inner_join(sperm_source, by = c("species", "sample")) |>
  is_sperm_row() |>
  group_by(species, sample, label, from, age, size) |>
  summarise(positions = sum(count), .groups = "drop") |>
  left_join(usable, by = c("species", "sample")) |>
  left_join(count(sperm_calls, species, sample, size, name = "indels"),
            by = c("species", "sample", "size")) |>
  mutate(indels        = coalesce(indels, 0L),
         # a sample with no calls at all has no reads to estimate its own share
         usable_share  = coalesce(usable_share, median(usable_share, na.rm = TRUE)),
         opportunities = positions * usable_share) |>
  add_rate()

cat(nrow(sperm_calls), "unique sperm calls beyond", read_end_cut, "bp from a read end, in",
    n_distinct(sperm_rates$label), "samples\n")
if (any(is.na(sperm_rates$age))) {
  cat("no age, so not in the plot:",
      paste(unique(sperm_rates$label[is.na(sperm_rates$age)]), collapse = ", "), "\n")
}

sperm_rates |>
  select(label, from, age, size, indels, opportunities, rate) |>
  mutate(opportunities = signif(opportunities, 3), rate = signif(rate, 3)) |>
  arrange(from, label, size) |>
  print(n = 60)

# The age trend, within the human sperm series. Quasi-Poisson on the counts with
# the read positions as offset, so the coefficient is a rate ratio per year and
# the dispersion is estimated rather than assumed.
# human/la4 has no calls at all and human/tsaed none that are unique. That is an
# upstream problem rather than a rate of zero, so they are dropped here.
no_calls <- sperm_rates |>
  group_by(label) |>
  filter(sum(indels) == 0) |>
  pull(label) |>
  unique()
if (length(no_calls) > 0) {
  cat("dropped, no unique calls at all:", paste(no_calls, collapse = ", "), "\n")
  sperm_rates <- filter(sperm_rates, !label %in% no_calls)
}

age_fit <- sperm_rates |>
  filter(from == "whole sample", !is.na(age)) |>
  group_by(size) |>
  group_modify(~ {
    m <- glm(indels ~ age + offset(log(opportunities)), family = quasipoisson, data = .x)
    s <- summary(m)$coefficients
    tibble(samples  = nrow(.x),
           per_year = round(exp(s["age", 1]), 4),
           ci_low   = round(exp(s["age", 1] - 1.96 * s["age", 2]), 4),
           ci_high  = round(exp(s["age", 1] + 1.96 * s["age", 2]), 4),
           p        = signif(s["age", 4], 2))
  }) |>
  ungroup()
cat("\nrate ratio per year of age, human sperm samples only:\n")
print(as.data.frame(age_fit), row.names = FALSE)

p_sperm_age <- sperm_rates |>
  filter(!is.na(age)) |>
  rate_points() |>
  ggplot(aes(x = age, y = shown)) +
  geom_smooth(data = ~ filter(.x, from == "whole sample", indels > 0),
              method = "lm", formula = y ~ x,
              colour = "grey30", fill = "grey80", linewidth = 0.5, alpha = 0.3) +
  geom_linerange(aes(ymin = bar_low, ymax = high, colour = species), linewidth = 0.4) +
  geom_point(aes(colour = species, shape = from, alpha = estimate), size = 2.2) +
  scale_alpha_manual(values = c("InDels observed" = 1, "none, upper bound" = 0.45), name = NULL) +
  ggrepel::geom_text_repel(aes(label = sample), size = 2.9, colour = "grey25",
                           min.segment.length = 0.2, seed = 1, max.overlaps = 20) +
  scale_shape_manual(values = c("whole sample" = 16, "sperm reads" = 17)) +
  scale_y_log10() +
  facet_wrap(~ size, nrow = 1, labeller = labeller(size = function(x) paste(x, "bp InDels"))) +
  labs(title = "Sperm InDel rate against age",
       x = "age of the animal (years)", y = "InDels per read position",
       colour = NULL, shape = NULL) +
  theme_minimal(base_size = 13) +
  theme(legend.position = "bottom", panel.grid.minor = element_blank(),
        plot.title.position = "plot", plot.title = element_text(hjust = 0.5))

options(repr.plot.width = 13, repr.plot.height = 6)
print(p_sperm_age)
options(repr.plot.width = 11, repr.plot.height = 6)

save_svg(p_sperm_age, "sperm_age.svg", width = 12, height = 6)

### The human sperm samples pooled, by context

All human samples under `sperm_samples` in config.yaml, their calls and their
read positions summed into one spectrum. `human/la4` has no calls at all and
`human/tsaed` none that are unique, so both are dropped rather than contributing
a denominator with an empty numerator.

Unique calls further than 300 bp from a read end, with each sample's read
positions scaled by the share surviving that cut before they are summed.
Deletions and insertions of the same sequence share one denominator, since an
opportunity is a read position where that k-mer could be gained or lost.

Pooling assumes the samples share a rate. They do not exactly — da1 sits highest
and the spread between them is visible in the age plot above — so the pooled
value is a read-weighted average, and the interval is narrower than the
sample-to-sample spread would justify.

In [ ]:
pool_runs <- runs |>
  filter(species == "human", is_sperm, has_calls, has_opportunities)

pool_calls <- indels |>
  semi_join(pool_runs, by = c("species", "sample")) |>
  filter(unique == "unique", ToreadEnd > read_end_cut)

# A sample contributing a denominator but no unique calls would drag the pooled
# rate down for a reason that is not biological.
no_calls <- setdiff(pool_runs$sample, unique(pool_calls$sample))
if (length(no_calls) > 0) {
  cat("dropped, no unique calls at all:", paste(no_calls, collapse = ", "), "\n")
  pool_runs <- filter(pool_runs, !sample %in% no_calls)
  pool_calls <- filter(pool_calls, !sample %in% no_calls)
}
cat("pooled:", paste(pool_runs$sample, collapse = ", "), "\n")

usable <- indels |>
  semi_join(pool_runs, by = c("species", "sample")) |>
  distinct(species, sample, readID, read_length) |>
  group_by(species, sample) |>
  summarise(usable_share = 1 - sum(pmin(2 * read_end_cut, read_length)) / sum(read_length),
            .groups = "drop")

pool_rates <- opportunities |>
  semi_join(pool_runs, by = c("species", "sample")) |>
  inner_join(usable, by = c("species", "sample")) |>
  mutate(positions = opportunities * usable_share) |>
  group_by(size, context) |>
  summarise(opportunities = sum(positions), .groups = "drop") |>
  cross_join(tibble(indel_type = c("deletion", "insertion"))) |>
  left_join(count(pool_calls, size, canonical_indel, indel_type, name = "indels"),
            by = c("size", "context" = "canonical_indel", "indel_type")) |>
  mutate(indels = coalesce(indels, 0L)) |>
  add_rate()

cat(sum(pool_rates$indels), "unique calls over",
    format(sum(distinct(pool_rates, size, context, opportunities)$opportunities), big.mark = ","),
    "read positions\n")

pool_rates |>
  select(size, context, indel_type, indels, opportunities, rate) |>
  mutate(opportunities = signif(opportunities, 3), rate = signif(rate, 3)) |>
  arrange(desc(rate)) |>
  print(n = 20)

p_sperm_context <- pool_rates |>
  rate_points() |>
  ggplot(aes(x = reorder(context, -rate, max), y = shown,
             colour = indel_type, group = indel_type)) +
  rate_layers() +
  scale_colour_manual(values = type_colours) +
  facet_grid(~ size, scales = "free_x", space = "free_x",
             labeller = labeller(size = function(x) paste(x, "bp"))) +
  labs(title = "Human sperm pooled, InDel rate by context",
       x = "context", y = "InDels per read position", colour = NULL) +
  theme_minimal(base_size = 13) +
  theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1),
        panel.grid.minor = element_blank(),
        legend.position = "bottom",
        plot.title.position = "plot", plot.title = element_text(hjust = 0.5))

options(repr.plot.width = 13, repr.plot.height = 6)
print(p_sperm_context)
options(repr.plot.width = 11, repr.plot.height = 6)

save_svg(p_sperm_context, "sperm_context.svg", width = 12, height = 6)

## The three groups side by side

One point per sample, coloured by group, so every sample in the results folder
appears exactly once. Samples in none of the three lists are shown as "other".

In [ ]:
group_rates <- size_rates |>
  group_by(label, group, size, unique) |>
  summarise(indels = sum(indels), opportunities = first(opportunities), .groups = "drop") |>
  add_rate() |>
  mutate(group = factor(group, levels = names(group_colours)))

options(repr.plot.width = 13, repr.plot.height = 7)
group_rates |>
  rate_points() |>
  ggplot(aes(x = group, y = shown, colour = group)) +
  geom_linerange(aes(ymin = bar_low, ymax = high),
                 position = position_jitter(width = 0.25, height = 0, seed = 1), linewidth = 0.4) +
  geom_point(aes(shape = estimate),
             position = position_jitter(width = 0.25, height = 0, seed = 1), size = 1.8) +
  scale_shape_manual(values = c("InDels observed" = 16, "none, upper bound" = 1), name = NULL) +
  scale_y_log10() +
  scale_colour_manual(values = group_colours, guide = "none") +
  facet_grid(unique ~ size, labeller = labeller(size = function(x) paste(x, "bp"))) +
  labs(title = "InDel rate by sample group",
       x = NULL, y = "calls per read position") +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))
options(repr.plot.width = 11, repr.plot.height = 6)

## Compared with the repeat models

The rate of 2 and 3 bp InDels outside repeats against the rate of one-copy
InDels inside M0 and M1 tracts of the same motif. Outside repeats, contexts are
grouped into motifs the way the tract motifs are (rotation and reverse
complement: AC and CA both belong to motif AC). Inside repeats, only InDels of
exactly one copy of the tract's motif are counted, so both sides count the same
kind of event. Unique calls only, on both sides.

In [ ]:
# The repeat calls and opportunities for the same samples, one-copy InDels at
# periods 2 and 3, with the same max_hap_distance as above.
repeat_runs <- expand_grid(distinct(runs, species, sample), model = models) |>
  mutate(dir      = file.path(repeat_indels_dir, species, sample, model),
         calls_ok = has_rows(file.path(dir, "indels.tsv")),
         opp_ok   = has_rows(file.path(dir, "opportunities.tsv"))) |>
  filter(calls_ok | opp_ok)

if (!any(repeat_runs$opp_ok)) {
  by_motif <- NULL
  message("No repeat opportunities for these samples under ", repeat_indels_dir)
} else {
  repeat_indels <- file.path(repeat_runs$dir[repeat_runs$calls_ok], "indels.tsv") |>
    lapply(read_calls) |>
    bind_rows() |>
    filter(k %in% c(2, 3), nchar(indel_string) == k,
           is.na(distance_to_indel_in_hap) | distance_to_indel_in_hap <= max_hap_distance,
           unique == "unique") |>
    filter(canonicalise(indel_string) == canonical_motif) |>
    count(species, sample, model, k, motif = canonical_motif, name = "indels")

  repeat_opportunities <- file.path(repeat_runs$dir[repeat_runs$opp_ok], "opportunities.tsv") |>
    lapply(function(path) {
      read_tsv(path, show_col_types = FALSE, progress = FALSE,
               col_types = cols(tissue = col_character())) |>
        filter(k %in% c(2, 3)) |>
        group_by(species, sample, k, motif = canonical_motif) |>
        summarise(opportunities = sum(count), .groups = "drop") |>
        mutate(model = basename(dirname(path)))
    }) |>
    bind_rows()

  cat("repeat results:", sum(repeat_runs$calls_ok), "model dirs with calls,",
      sum(repeat_runs$opp_ok), "with opportunities\n")

  inside <- repeat_opportunities |>
    left_join(repeat_indels, by = c("species", "sample", "model", "k", "motif")) |>
    mutate(indels = coalesce(indels, 0L), source = model)

  # Outside repeats, grouped by motif. Homopolymer motifs (AA, CCC, ...) have no
  # counterpart among the tracts, which start at k = 2.
  motif_of <- distinct(opportunities, context) |>
    mutate(motif = canonicalise(context))

  outside <- opportunities |>
    filter(size %in% c(2, 3)) |>
    left_join(motif_of, by = "context") |>
    group_by(species, sample, k = size, motif) |>
    summarise(opportunities = sum(opportunities), .groups = "drop") |>
    left_join(indels |>
                filter(size %in% c(2, 3), unique == "unique") |>
                left_join(motif_of, by = c("canonical_indel" = "context")) |>
                count(species, sample, k = size, motif, name = "indels"),
              by = c("species", "sample", "k", "motif")) |>
    mutate(indels = coalesce(indels, 0L), source = "outside")

  # Homopolymer motifs are left out, and so are the few M1 motifs that do not
  # have k bases (TRF's consensus can differ in length from its period).
  by_motif <- bind_rows(outside, select(inside, -model)) |>
    filter(nchar(motif) == k,
           vapply(strsplit(motif, ""), function(b) length(unique(b)) > 1, logical(1))) |>
    mutate(source = factor(source, levels = c("outside", models)),
           label  = paste(species, sample, sep = " / ")) |>
    add_rate()

  by_motif |>
    group_by(label, k, source) |>
    summarise(indels = sum(indels), opportunities = sum(opportunities), .groups = "drop") |>
    add_rate() |>
    select(label, k, source, indels, opportunities, rate) |>
    print(n = 60)
}

In [ ]:
if (!is.null(by_motif)) {
  source_labels <- c(outside = "outside repeats", setNames(paste(models, "tracts"), models))

  # Motifs a model has almost no tracts of give intervals spanning orders of
  # magnitude, so they are left out of the plot.
  min_opportunities_shown <- 1e5

  options(repr.plot.width = 13, repr.plot.height = facet_height(n_distinct(by_motif$label), per_row = 1, row_height = 1.8))
  print(by_motif |>
    filter(opportunities >= min_opportunities_shown) |>
    rate_points() |>
    ggplot(aes(x = reorder(motif, -rate, max), y = shown, colour = source, group = source)) +
    rate_layers(dodge_width = 0.7) +
    scale_colour_manual(values = model_colours, labels = source_labels) +
    facet_grid(label ~ k, scales = "free_x", space = "free_x",
               labeller = labeller(k = function(x) paste0(x, " bp / k = ", x))) +
    labs(title = "One-copy InDel rate by motif",
         x = "motif", y = "InDels per read base", colour = NULL) +
    theme_minimal() +
    theme(axis.text.x = element_text(angle = 45, hjust = 1),
          strip.text.y = element_text(angle = 0, hjust = 0)))
  options(repr.plot.width = 11, repr.plot.height = 6)
}